# Session 1 — The Refactored ML Pipeline

**Goal:** Walk through how the messy notebook becomes a reusable, testable system.

Key changes:
- **Config file** (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">config.yml</span>) replaces magic numbers
- **`churn_model` package** replaces inline code
- **MLflow** replaces print statements and pickle files
- **Train/serving parity** — the same <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">features.py</span> is used at both train and serve time

In [0]:
%run ../utils/config

## Best Practice #1: Package your code into Python packages
A best practice is to package and use your reusable code in whl files, which are built packages.  In this notebook, we will install the package from the source files on the file system so we can browse the source and see what's being done.  A whl file is the same code bundled into a single file, making it portable and versionable.  Installing from a whl file would look like the following:

`%pip install churn_model-0.1.0-py3-none-any.whl --quiet`

In [0]:
%pip install ../bundle/wheels/ --quiet

### What are Python packages?
A deep treatment of the code is beyond the scope of this class, but feel free to browse the source code that we will use to train, evaluate, and invoke predictions with our models.  The code is organized into four **modules** within the `churn_model` **package**:
-  `evaluate`
-  `features`
-  `predict` and
-  `train`

Each module defines **functions** within it.  For example, the `features` module includes the following functions:
-  <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">build_feature_pipeline(config: dict) </span>
-  <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">prepare_dataframe(df: pd.DataFrame, config: dict)</span>
-  <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">run_data_quality_checks(df: pd.DataFrame, config: dict)</span>

You will use some of these functions in this notebook and those that follow.

Feel free to browse the source code of the package by navigating to <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">../bundle/wheels/churn_model</span>.

## Best Practice #2:  Load settings from a config file
Instead of hard-coded numbers and strings scattered throughout the notebook, all configuration lives in one YAML file.

This makes it easy to:

- Version control your hyperparameter choices
- Override values for different experiments
- Share config between the notebook and the production job

You can review the config file by navigating to <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">../common/config.yml</span>.

This link opens the file in a new browser tab:
[config.yml]($../common/config.yml)

In [0]:
import yaml

# Load the shared config file
config_path = "../common/config.yml"

with open(config_path) as f:
    config = yaml.safe_load(f)

print("Config loaded successfully!")
print("\nSome selected configuration values:")
print(f"  Target column  : {config['target_column']}")
print(f"  Numeric features : {config['feature_columns']['numeric']}")
print(f"  Model types    : {list(config['models'].keys())}")
print(f"  Primary metric : {config['training']['primary_metric']}")

## Step 1: Feature Pipeline

The <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">build_feature_pipeline()</span> function in <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_model/features.py</span> creates a sklearn `ColumnTransformer` that:
- Imputes missing `TotalCharges` values with the median
- Scales numeric features
- One-hot encodes all categorical features

**Critical:** This same function is called when the model runs in Model Serving.
There is no second copy of the feature logic — no opportunity for skew.

1.  Right-click on the function name <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">build_feature_pipeline</span> in the cell below and choose `Peek Definition` to see what the code does. _**Note**: If `Peek Definition` does not work, refresh your browser page._

In [0]:
from churn_model.features import build_feature_pipeline

preprocessor = build_feature_pipeline(config)
print(preprocessor)
print("\nTransformers:")
for name, transformer, cols in preprocessor.transformers:
    print(f"  [{name}] → {cols}")

## Step 2: Train a Single Model

Run the next cell to train a Random Forest model.  

Key points:
-  Configuration settings are pulled from the `config.yml` file read above.
-  Warnings are being supressed to make the output cleaner, but you should always investigate warnings when you begin a project
- The code uses the `run_training()` function in the `train` module of the `churn_model` imported above.
-  `run_training()` uses MLflow to track everything automatically.

In [0]:
import warnings
from churn_model.train import run_training

# Load data from the shared table for this demo run.
# In the full pipeline (04_mlflow_experiment) we use the Feature Store;
# here we pass the DataFrame directly so this notebook stays self-contained.
df = spark.table(f"{catalog}.`00_shared`.telco_churn").toPandas()

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")
    result = run_training(
        catalog=catalog,
        schema=schema,
        config=config,
        model_type="random_forest",
        data=df,
    )

After the cell finishes running, view the MLflow experiment.

**In the UI:**
1. Open **Experiments** from the left sidebar in a new browser tab
2. Find your <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_{safe_username}</span> experiment.  This experiment is stored in the workspace filesystem under `/Users/{your_email}/churn_{safe_username}`, but can easily be accessed through the Experiments sidebar option.
3. Click your run and explore the logged params, metrics, and model artifacts

<img src=../utils/resources/FirstExperimentRun.png>

## Step 3: Manual test of the model

Load the model from the MLflow experiment, then run a prediction against a test customer record.

In [0]:
import mlflow, pandas as pd
import numpy as np

# Load the model that was just trained
model = mlflow.pyfunc.load_model(f"runs:/{result.run_id}/model")

# One customer that we would expect to be high-risk: month-to-month, fiber optic, short tenure
test_record = pd.DataFrame([{
    "tenure":           np.int32(2),
    "MonthlyCharges":   95.50,
    "TotalCharges":     191.00,
    "gender":           "Female",
    "SeniorCitizen":    "0",
    "Partner":          "No",
    "Dependents":       "No",
    "PhoneService":     "Yes",
    "MultipleLines":    "No",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "No",
    "OnlineBackup":     "No",
    "DeviceProtection": "No",
    "TechSupport":      "No",
    "StreamingTV":      "No",
    "StreamingMovies":  "No",
    "Contract":         "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Electronic check",
}])

prediction = model.predict(test_record)

print(f"Prediction for fake high-risk customer: {prediction[0]}\n")
print(f"  (expected: 'Yes' (1) - short tenure, fiber optic, month-to-month, no add-ons)")

## What Did MLflow Track?

You should see your run with:
- **Parameters**: model_type, n_estimators, max_depth, test_size, random_state
- **Metrics**: test_f1, test_roc_auc, test_precision, test_recall, test_accuracy
- **Artifacts**: the trained model, classification_report.txt

**Discussion**: What's the value of tracking all of this automatically?
How does this change your workflow when you come back to this model 3 months later?

➡️ Next: [04_mlflow_experiment.ipynb]($./04_mlflow_experiment) — compare three challenger models side by side